In [ ]:
# CARABAS VFH Change Detection - Full Paper Implementation
# This script implements the Neyman-Pearson criterion-based change detection methods for wavelength-resolution SAR image stacks as described in the referenced research paper.
# It uses block-based Rician background modeling, saves parameter maps for reuse, and generates ROC curves for both NPCBS and NPC methods.

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rice, uniform
from scipy import ndimage
import time
import os

In [ ]:
# --- 1. SETUP: Paths, Metadata, and Helper Functions ---
IMAGE_DIR = '../images'
TARGET_DIR = '../target_lists'
PARAM_DIR = 'parameter_maps_6x9'  # Directory to save/load MLE results

if not os.path.exists(PARAM_DIR):
    os.makedirs(PARAM_DIR)

image_metadata = [
    # ... (same as in notebook, omitted for brevity) ...
]

epsilon = 1e-9

In [ ]:
def load_sar_image(filename, directory, rows=3000, cols=2000):
    filepath = os.path.join(directory, filename)
    image_data = np.fromfile(filepath, dtype='>f4').reshape((rows, cols))
    return image_data

In [ ]:
def load_ground_truth_pixels(deployment_name, target_dir):
    target_filename = deployment_name + ".Targets.txt"
    target_filepath = os.path.join(target_dir, target_filename)
    ground_truth_geo = np.loadtxt(target_filepath, usecols=(0, 1))
    Nmax, Emin = 7370488, 1653166
    ground_truth_pixels = []
    for north, east in ground_truth_geo:
        row = int(round(Nmax - north))
        col = int(round(east - Emin))
        ground_truth_pixels.append((row, col))
    return ground_truth_pixels

In [ ]:
def evaluate_performance(detection_map, ground_truth_pixels):
    eroded_map = ndimage.binary_erosion(detection_map)
    processed_map = ndimage.binary_dilation(eroded_map, iterations=2)
    labeled_map, num_objects = ndimage.label(processed_map)
    if num_objects == 0:
        return 0.0, 0.0
    detected_centers = ndimage.center_of_mass(processed_map, labeled_map, range(1, num_objects + 1))
    match_radius = 10.0
    true_positives = 0
    unmatched_detections = list(detected_centers)
    num_true_targets = len(ground_truth_pixels)
    for true_target_pos in ground_truth_pixels:
        for i, detected_pos in enumerate(unmatched_detections):
            distance = np.sqrt((true_target_pos[0] - detected_pos[0])**2 + (true_target_pos[1] - detected_pos[1])**2)
            if distance <= match_radius:
                true_positives += 1
                unmatched_detections.pop(i)
                break
    false_positives = len(unmatched_detections)
    image_area_km2 = 6.0
    pd = true_positives / num_true_targets if num_true_targets > 0 else 0.0
    far = false_positives / image_area_km2
    return pd, far

In [ ]:
# --- 2. Data Preparation ---
surveillance_info = image_metadata[0]
background_stack_info = [img for img in image_metadata if img['heading'] == surveillance_info['heading'] and img['deployment'] != surveillance_info['deployment']]
surveillance_image = load_sar_image(surveillance_info['filename'], IMAGE_DIR)
background_images = [load_sar_image(info['filename'], IMAGE_DIR) for info in background_stack_info]
background_stack = np.stack(background_images, axis=0)
ground_truth = load_ground_truth_pixels(surveillance_info['deployment'], TARGET_DIR)

In [ ]:
# --- 3. Rician Background Modeling (Fixed, Non-Overlapping Blocks) ---
B_MAP_FILE = os.path.join(PARAM_DIR, 'b_map_fixed.npy')
LOC_MAP_FILE = os.path.join(PARAM_DIR, 'loc_map_fixed.npy')
SCALE_MAP_FILE = os.path.join(PARAM_DIR, 'scale_map_fixed.npy')

if os.path.exists(B_MAP_FILE):
    print("Loading pre-calculated fixed-block Rician parameter maps...")
    b_map = np.load(B_MAP_FILE)
    loc_map = np.load(LOC_MAP_FILE)
    scale_map = np.load(SCALE_MAP_FILE)
else:
    print("Performing fixed-block Rician MLE fit. This will be much faster than sliding window.")
    start_time = time.time()
    rows, cols = surveillance_image.shape
    b_map = np.zeros((rows, cols))
    loc_map = np.zeros((rows, cols))
    scale_map = np.zeros((rows, cols))
    BLOCK_WIDTH = 6
    BLOCK_HEIGHT = 9
    for r_start in range(0, rows, BLOCK_HEIGHT):
        for c_start in range(0, cols, BLOCK_WIDTH):
            r_end = min(r_start + BLOCK_HEIGHT, rows)
            c_end = min(c_start + BLOCK_WIDTH, cols)
            block_data = background_stack[:, r_start:r_end, c_start:c_end]
            samples = block_data.flatten()
            try:
                b, loc, scale = rice.fit(samples, floc=0)
            except Exception as e:
                mean = np.mean(samples)
                std = np.std(samples)
                b = mean / std if std > 0 else 0
                loc = 0
                scale = std if std > 0 else epsilon
            b_map[r_start:r_end, c_start:c_end] = b
            loc_map[r_start:r_end, c_start:c_end] = loc
            scale_map[r_start:r_end, c_start:c_end] = scale
        if (r_start + BLOCK_HEIGHT) % 100 < BLOCK_HEIGHT:
            elapsed = time.time() - start_time
            print(f"Processed up to row {r_start + BLOCK_HEIGHT}/{rows}... Time: {elapsed/60:.2f} min.")
    print("Fixed-block MLE fitting complete. Saving maps to disk...")
    np.save(B_MAP_FILE, b_map)
    np.save(LOC_MAP_FILE, loc_map)
    np.save(SCALE_MAP_FILE, scale_map)
print("Background model is ready.\n")

In [ ]:
# --- 4. ROC Curve Generation ---
pfa_thresholds = [0.5, 0.2, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]
tau_thresholds = [1, 10, 100, 1000, 1e4, 1e5, 1e6, 1e7]
npcbs_results = []
npc_results = []
print("Running NPCBS method...")
for p_fa in pfa_thresholds:
    detection_threshold = 1 - p_fa
    cdf_values = rice.cdf(surveillance_image, b_map, loc=loc_map, scale=(scale_map + epsilon))
    detection_map = cdf_values >= detection_threshold
    pd, far = evaluate_performance(detection_map, ground_truth)
    npcbs_results.append((far, pd))
    print(f"  P_FA={p_fa:.1e} -> Pd={pd:.2%}, FAR={far:.2f}")
print("\nRunning NPC method...")
a_min, a_max = 0.4, np.max(surveillance_image)
p_background = rice.pdf(surveillance_image, b_map, loc=loc_map, scale=(scale_map + epsilon))
p_target = uniform.pdf(surveillance_image, loc=a_min, scale=(a_max - a_min))
likelihood_ratio = p_target / (p_background + epsilon)
for tau in tau_thresholds:
    detection_map = likelihood_ratio >= tau
    pd, far = evaluate_performance(detection_map, ground_truth)
    npc_results.append((far, pd))
    print(f"  Tau={tau:.0e} -> Pd={pd:.2%}, FAR={far:.2f}")

In [ ]:
# --- 5. Plotting the ROC Curve ---
plt.figure(figsize=(10, 7))
npcbs_results.sort()
npc_results.sort()
if npcbs_results:
    npcbs_far, npcbs_pd = zip(*npcbs_results)
    plt.plot(npcbs_far, npcbs_pd, 'o-', label='NPCBS Method')
if npc_results:
    npc_far, npc_pd = zip(*npc_results)
    plt.plot(npc_far, npc_pd, 's-', label='NPC Method (a_min=0.4)')
plt.xscale('log')
plt.grid(True, which="both", ls="--")
plt.xlabel('False Alarm Rate (FAR) [alarms/km²]')
plt.ylabel('Probability of Detection (Pd)')
plt.title('ROC Curve Comparison (Block-Based MLE)')
plt.legend(loc='lower right')
plt.ylim(0, 1.05)
plt.xlim(1e-2, 1e2)
plt.show()